In [1]:
import duckdb
import pandas as pd
import os

os.chdir(r'C:\Users\Tobi\Desktop\CSHW\TransitKPIFramework')
con = duckdb.connect('data/transit_kpi.duckdb')

print(con.sql("SHOW TABLES").df().to_string())

             name
0        calendar
1  calendar_dates
2          gtfsrt
3      stop_times
4     trip_delays
5           trips


In [2]:
con.execute("DROP TABLE IF EXISTS trip_delays")

con.execute("""
    CREATE TABLE trip_delays AS
    WITH active_services AS (
        SELECT service_id FROM calendar
        WHERE tuesday = 1
        AND start_date <= 20260407
        AND end_date >= 20260407

        UNION

        SELECT service_id FROM calendar_dates
        WHERE date = 20260407
        AND exception_type = 1
    ),
    matched AS (
        SELECT
            rt.trip_id,
            rt.route_id,
            rt.service_date,
            rt.snapshot_ts,
            rt.stop_sequence,
            rt.stop_id,
            rt.schedule_relationship,
            rt.predicted_unix,
            rt.predicted_ts,
            rt.midnight_unix,
            st.arrival_time                 AS scheduled_arrival_time,
            st.arrival_seconds              AS scheduled_arrival_seconds,
            -- Add timepoint flag and departure time
            st.timepoint,
            st.departure_time,
            CAST(SPLIT_PART(st.departure_time, ':', 1) AS INTEGER) * 3600 +
            CAST(SPLIT_PART(st.departure_time, ':', 2) AS INTEGER) * 60 +
            CAST(SPLIT_PART(st.departure_time, ':', 3) AS INTEGER) AS departure_seconds
        FROM gtfsrt rt
        JOIN trips t
            ON rt.trip_id = t.trip_id
        JOIN active_services a
            ON t.service_id = a.service_id
        JOIN stop_times st
            ON rt.trip_id        = st.trip_id
            AND rt.stop_sequence = st.stop_sequence
        WHERE rt.route_id IN ('23', '47')
        AND rt.schedule_relationship = 0
    ),
    nearest_day AS (
        SELECT
            *,
            CASE
                WHEN ABS(predicted_unix - (midnight_unix + scheduled_arrival_seconds - 86400))
                   < ABS(predicted_unix - (midnight_unix + scheduled_arrival_seconds))
                 AND ABS(predicted_unix - (midnight_unix + scheduled_arrival_seconds - 86400))
                   < ABS(predicted_unix - (midnight_unix + scheduled_arrival_seconds + 86400))
                THEN midnight_unix + scheduled_arrival_seconds - 86400
                WHEN ABS(predicted_unix - (midnight_unix + scheduled_arrival_seconds + 86400))
                   < ABS(predicted_unix - (midnight_unix + scheduled_arrival_seconds))
                THEN midnight_unix + scheduled_arrival_seconds + 86400
                ELSE midnight_unix + scheduled_arrival_seconds
            END AS scheduled_unix
        FROM matched
    ),
    deduped AS (
        SELECT
            *,
            ROUND((predicted_unix - scheduled_unix) / 60.0, 1) AS delay_minutes,
            ROW_NUMBER() OVER (
                PARTITION BY trip_id, stop_sequence
                ORDER BY ABS(predicted_unix - scheduled_unix)
            ) AS snapshot_rank
        FROM nearest_day
    )
    SELECT
        trip_id,
        route_id,
        service_date,
        snapshot_ts,
        stop_sequence,
        stop_id,
        schedule_relationship,
        scheduled_arrival_time,
        scheduled_arrival_seconds,
        departure_time,
        departure_seconds,
        timepoint,
        scheduled_unix,
        predicted_unix,
        predicted_ts,
        delay_minutes
    FROM deduped
    WHERE snapshot_rank = 1
""")

print(con.sql("SELECT COUNT(*) as rows FROM trip_delays").df().to_string())

    rows
0  42041


In [3]:
print(con.sql("""
    SELECT
        route_id,
        COUNT(*)                                                          AS total_stops,
        -- Strict: 0 to +3 min
        ROUND(SUM(CASE WHEN delay_minutes BETWEEN 0 AND 3
                       THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1)        AS otp_strict,
        -- Standard: -1 to +5 min
        ROUND(SUM(CASE WHEN delay_minutes BETWEEN -1 AND 5
                       THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1)        AS otp_standard,
        -- Lenient: -2 to +7 min
        ROUND(SUM(CASE WHEN delay_minutes BETWEEN -2 AND 7
                       THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1)        AS otp_lenient
    FROM trip_delays
    GROUP BY route_id
    ORDER BY route_id
""").df().to_string())

  route_id  total_stops  otp_strict  otp_standard  otp_lenient
0       23        21706        49.2          85.9         94.5
1       47        20335        44.3          79.8         91.3


In [4]:
print(con.sql("""
    SELECT
        route_id,
        COUNT(*)                                                          AS total_stops,
        -- No early tolerance: must not depart early at all
        ROUND(SUM(CASE WHEN delay_minutes BETWEEN 0 AND 5
                       THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1)        AS otp_no_early,
        -- Standard: up to 1 min early ok
        ROUND(SUM(CASE WHEN delay_minutes BETWEEN -1 AND 5
                       THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1)        AS otp_1min_early,
        -- Lenient: any early departure ok
        ROUND(SUM(CASE WHEN delay_minutes <= 5
                       THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1)        AS otp_any_early
    FROM trip_delays
    GROUP BY route_id
    ORDER BY route_id
""").df().to_string())

  route_id  total_stops  otp_no_early  otp_1min_early  otp_any_early
0       23        21706          55.5            85.9           95.4
1       47        20335          47.9            79.8           91.3


In [5]:
print(con.sql("""
    SELECT
        route_id,
        -- All stops
        ROUND(SUM(CASE WHEN delay_minutes BETWEEN -1 AND 5
                       THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1)        AS otp_all_stops,
        -- Timepoints only
        ROUND(SUM(CASE WHEN delay_minutes BETWEEN -1 AND 5
                            AND timepoint = 1
                       THEN 1 ELSE 0 END) * 100.0 /
              NULLIF(SUM(CASE WHEN timepoint = 1 THEN 1 ELSE 0 END), 0)
        , 1)                                                             AS otp_timepoints_only,
        -- First and last stop of each trip
        ROUND(SUM(CASE WHEN delay_minutes BETWEEN -1 AND 5
                            AND stop_sequence IN (
                                SELECT MIN(stop_sequence) FROM trip_delays td2
                                WHERE td2.trip_id = trip_delays.trip_id
                                UNION
                                SELECT MAX(stop_sequence) FROM trip_delays td2
                                WHERE td2.trip_id = trip_delays.trip_id
                            )
                       THEN 1 ELSE 0 END) * 100.0 /
              NULLIF(SUM(CASE WHEN stop_sequence IN (
                                SELECT MIN(stop_sequence) FROM trip_delays td2
                                WHERE td2.trip_id = trip_delays.trip_id
                                UNION
                                SELECT MAX(stop_sequence) FROM trip_delays td2
                                WHERE td2.trip_id = trip_delays.trip_id
                            ) THEN 1 ELSE 0 END), 0)
        , 1)                                                             AS otp_terminals_only
    FROM trip_delays
    GROUP BY route_id
    ORDER BY route_id
""").df().to_string())

  route_id  otp_all_stops  otp_timepoints_only  otp_terminals_only
0       23           85.9                 83.9                83.5
1       47           79.8                 79.6                76.9


In [6]:
print(con.sql("""
    SELECT
        route_id,
        -- Time of day bucket based on scheduled arrival (Eastern = seconds since midnight)
        CASE
            WHEN scheduled_arrival_seconds BETWEEN 25200 AND 32400 THEN 'AM Peak (7-9am)'
            WHEN scheduled_arrival_seconds BETWEEN 32400 AND 57600 THEN 'Midday (9am-4pm)'
            WHEN scheduled_arrival_seconds BETWEEN 57600 AND 68400 THEN 'PM Peak (4-7pm)'
            ELSE                                                         'Off Peak'
        END                                                              AS time_period,
        COUNT(*)                                                         AS total_stops,
        ROUND(SUM(CASE WHEN delay_minutes BETWEEN -1 AND 5
                       THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1)        AS otp_pct
    FROM trip_delays
    GROUP BY route_id, time_period
    ORDER BY route_id, MIN(scheduled_arrival_seconds)
""").df().to_string())

  route_id       time_period  total_stops  otp_pct
0       23          Off Peak         6762     89.3
1       23   AM Peak (7-9am)         2639     90.1
2       23  Midday (9am-4pm)         8766     83.5
3       23   PM Peak (4-7pm)         3539     82.5
4       47          Off Peak         6086     85.5
5       47   AM Peak (7-9am)         2582     78.8
6       47  Midday (9am-4pm)         8000     82.8
7       47   PM Peak (4-7pm)         3667     64.5


In [7]:
print(con.sql("""
    WITH stop_position AS (
        SELECT
            trip_id,
            stop_sequence,
            route_id,
            delay_minutes,
            -- Calculate what percentile of the route this stop is
            ROUND(
                (stop_sequence - MIN(stop_sequence) OVER (PARTITION BY trip_id)) * 100.0 /
                NULLIF(MAX(stop_sequence) OVER (PARTITION BY trip_id) -
                       MIN(stop_sequence) OVER (PARTITION BY trip_id), 0)
            , 0) AS pct_through_route
        FROM trip_delays
    )
    SELECT
        route_id,
        CASE
            WHEN pct_through_route <= 25  THEN 'First quarter'
            WHEN pct_through_route <= 50  THEN 'Second quarter'
            WHEN pct_through_route <= 75  THEN 'Third quarter'
            ELSE                               'Final quarter'
        END                                                              AS route_position,
        COUNT(*)                                                         AS total_stops,
        ROUND(SUM(CASE WHEN delay_minutes BETWEEN -1 AND 5
                       THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1)        AS otp_pct,
        ROUND(AVG(delay_minutes), 1)                                     AS avg_delay_min
    FROM stop_position
    GROUP BY route_id, route_position
    ORDER BY route_id, MIN(pct_through_route)
""").df().to_string())

  route_id  route_position  total_stops  otp_pct  avg_delay_min
0       23   First quarter         5586     86.6            1.2
1       23  Second quarter         5378     86.1            0.3
2       23   Third quarter         5377     88.9            0.5
3       23   Final quarter         5365     82.2            0.9
4       47   First quarter         5200     79.1            1.7
5       47  Second quarter         5047     83.0            0.8
6       47   Third quarter         5017     80.3            0.6
7       47   Final quarter         5071     76.8            1.7


In [8]:
print(con.sql("""
    SELECT
        'Lateness threshold'        AS dimension,
        'Strict (0 to +3)'         AS specification,
        route_id,
        ROUND(SUM(CASE WHEN delay_minutes BETWEEN 0 AND 3
                       THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS otp_pct
    FROM trip_delays GROUP BY route_id

    UNION ALL

    SELECT 'Lateness threshold', 'Standard (-1 to +5)', route_id,
        ROUND(SUM(CASE WHEN delay_minutes BETWEEN -1 AND 5
                       THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1)
    FROM trip_delays GROUP BY route_id

    UNION ALL

    SELECT 'Lateness threshold', 'Lenient (-2 to +7)', route_id,
        ROUND(SUM(CASE WHEN delay_minutes BETWEEN -2 AND 7
                       THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1)
    FROM trip_delays GROUP BY route_id

    UNION ALL

    SELECT 'Early departure', 'No early allowed', route_id,
        ROUND(SUM(CASE WHEN delay_minutes BETWEEN 0 AND 5
                       THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1)
    FROM trip_delays GROUP BY route_id

    UNION ALL

    SELECT 'Early departure', 'Any early ok', route_id,
        ROUND(SUM(CASE WHEN delay_minutes <= 5
                       THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1)
    FROM trip_delays GROUP BY route_id

    UNION ALL

    SELECT 'Stop selection', 'All stops', route_id,
        ROUND(SUM(CASE WHEN delay_minutes BETWEEN -1 AND 5
                       THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1)
    FROM trip_delays GROUP BY route_id

    UNION ALL

    SELECT 'Stop selection', 'Timepoints only', route_id,
        ROUND(SUM(CASE WHEN delay_minutes BETWEEN -1 AND 5 AND timepoint = 1
                       THEN 1 ELSE 0 END) * 100.0 /
              NULLIF(SUM(CASE WHEN timepoint = 1 THEN 1 ELSE 0 END), 0), 1)
    FROM trip_delays GROUP BY route_id

    ORDER BY dimension, specification, route_id
""").df().to_string())

             dimension        specification route_id  otp_pct
0      Early departure         Any early ok       23     95.4
1      Early departure         Any early ok       47     91.3
2      Early departure     No early allowed       23     55.5
3      Early departure     No early allowed       47     47.9
4   Lateness threshold   Lenient (-2 to +7)       23     94.5
5   Lateness threshold   Lenient (-2 to +7)       47     91.3
6   Lateness threshold  Standard (-1 to +5)       23     85.9
7   Lateness threshold  Standard (-1 to +5)       47     79.8
8   Lateness threshold     Strict (0 to +3)       23     49.2
9   Lateness threshold     Strict (0 to +3)       47     44.3
10      Stop selection            All stops       23     85.9
11      Stop selection            All stops       47     79.8
12      Stop selection      Timepoints only       23     83.9
13      Stop selection      Timepoints only       47     79.6


In [9]:
con.execute("DROP TABLE IF EXISTS otp_specifications")

con.execute("""
    CREATE TABLE otp_specifications AS

    SELECT 'Lateness threshold' AS dimension, 'Strict (0 to +3)' AS specification, route_id,
        ROUND(SUM(CASE WHEN delay_minutes BETWEEN 0 AND 3 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS otp_pct
    FROM trip_delays GROUP BY route_id

    UNION ALL

    SELECT 'Lateness threshold', 'Standard (-1 to +5)', route_id,
        ROUND(SUM(CASE WHEN delay_minutes BETWEEN -1 AND 5 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1)
    FROM trip_delays GROUP BY route_id

    UNION ALL

    SELECT 'Lateness threshold', 'Lenient (-2 to +7)', route_id,
        ROUND(SUM(CASE WHEN delay_minutes BETWEEN -2 AND 7 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1)
    FROM trip_delays GROUP BY route_id

    UNION ALL

    SELECT 'Early departure', 'No early allowed', route_id,
        ROUND(SUM(CASE WHEN delay_minutes BETWEEN 0 AND 5 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1)
    FROM trip_delays GROUP BY route_id

    UNION ALL

    SELECT 'Early departure', 'Standard (1 min early)', route_id,
        ROUND(SUM(CASE WHEN delay_minutes BETWEEN -1 AND 5 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1)
    FROM trip_delays GROUP BY route_id

    UNION ALL

    SELECT 'Early departure', 'Any early ok', route_id,
        ROUND(SUM(CASE WHEN delay_minutes <= 5 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1)
    FROM trip_delays GROUP BY route_id

    UNION ALL

    SELECT 'Stop selection', 'All stops', route_id,
        ROUND(SUM(CASE WHEN delay_minutes BETWEEN -1 AND 5 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1)
    FROM trip_delays GROUP BY route_id

    UNION ALL

    SELECT 'Stop selection', 'Timepoints only', route_id,
        ROUND(SUM(CASE WHEN delay_minutes BETWEEN -1 AND 5 AND timepoint = 1 THEN 1 ELSE 0 END) * 100.0 /
              NULLIF(SUM(CASE WHEN timepoint = 1 THEN 1 ELSE 0 END), 0), 1)
    FROM trip_delays GROUP BY route_id

    UNION ALL

    SELECT 'Time of day', 'AM Peak (7-9am)', route_id,
        ROUND(SUM(CASE WHEN delay_minutes BETWEEN -1 AND 5
                        AND scheduled_arrival_seconds BETWEEN 25200 AND 32400 THEN 1 ELSE 0 END) * 100.0 /
              NULLIF(SUM(CASE WHEN scheduled_arrival_seconds BETWEEN 25200 AND 32400 THEN 1 ELSE 0 END), 0), 1)
    FROM trip_delays GROUP BY route_id

    UNION ALL

    SELECT 'Time of day', 'Midday (9am-4pm)', route_id,
        ROUND(SUM(CASE WHEN delay_minutes BETWEEN -1 AND 5
                        AND scheduled_arrival_seconds BETWEEN 32400 AND 57600 THEN 1 ELSE 0 END) * 100.0 /
              NULLIF(SUM(CASE WHEN scheduled_arrival_seconds BETWEEN 32400 AND 57600 THEN 1 ELSE 0 END), 0), 1)
    FROM trip_delays GROUP BY route_id

    UNION ALL

    SELECT 'Time of day', 'PM Peak (4-7pm)', route_id,
        ROUND(SUM(CASE WHEN delay_minutes BETWEEN -1 AND 5
                        AND scheduled_arrival_seconds BETWEEN 57600 AND 68400 THEN 1 ELSE 0 END) * 100.0 /
              NULLIF(SUM(CASE WHEN scheduled_arrival_seconds BETWEEN 57600 AND 68400 THEN 1 ELSE 0 END), 0), 1)
    FROM trip_delays GROUP BY route_id
""")

print(con.sql("SELECT COUNT(*) as rows FROM otp_specifications").df().to_string())

   rows
0    22
